<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Potencial exportador por producto y mercado de destino

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

Define la carpeta donde queda la descarga del Export Potential Map del Centro de Comercio
Internacional, que produce `00_download/03_potencial_exportador_itc`, y las equivalencias de
nombre que hacen falta para asignarle a cada mercado su código ISO3.

**Nota de migración:** hasta esta versión el notebook leía 190 archivos de valores separados por
comas descargados a mano del mismo portal, en `data/raw/potencial_exportador/country_data/`. Esa
carpeta se conserva como archivo. El cambio a la descarga automatizada altera las cifras: el
portal revisó sus estimaciones. Ver `tools/markdown/AUDITORIA_MIGRACION.md`.

In [3]:
carpeta_itc = raw / 'itc'

# El portal y la tabla ISO nombran cuatro mercados de forma distinta. Sin estos alias,
# dos de ellos quedarian sin codigo y se perderian; los otros dos son mercados que la
# descarga trae y los archivos anteriores no tenian
alias_mercados = {
    'Chinese Taipei': 'Taipei, Chinese',
    'Hong Kong, China': 'Hong Kong SAR',
}

# mercados que la tabla ISO no lista
iso_adicionales = {
    'Bhutan': 'BTN',
    'Saint Pierre and Miquelon': 'SPM',
}

### Lectura de datos

Toma la descarga más reciente del portal, la tabla de correspondencia entre nomenclaturas
arancelarias y la equivalencia entre nombre de mercado y código ISO3.

In [4]:
archivos_itc = sorted(carpeta_itc.glob('colombia_export_potential_*.parquet'))
archivo_itc = archivos_itc[-1]   # la descarga mas reciente

df_potencial = pd.read_parquet(archivo_itc)
df_hs2012_2017 = pd.read_parquet(temp / 'df_hs2012_2017.parquet')
df_iso = pd.read_excel(raw / 'potencial_exportador' / 'ISO ITC Treemap.xlsx').rename(
    columns={'ISO3': 'ISO Importador', 'Country': 'market_name'})

print(f"{archivo_itc.name}: {len(df_potencial):,} filas, {df_potencial['market_code'].nunique()} mercados")

colombia_export_potential_2026-09-19.parquet: 179,889 filas, 189 mercados


### Consolidación del potencial exportador

**Pasos realizados:**

1. **Códigos de país:** normaliza los nombres que el portal escribe distinto de la tabla ISO, traduce el mercado de destino a su código ISO3 y fija a Colombia como exportador.

2. **Traducción arancelaria:** lleva el código del portal, que está en la nomenclatura HS 2017, a la HS 2012 que usa el resto de la cadena. Los grupos agregados del portal, que terminan en `XX` o vienen a cuatro dígitos, no tienen correspondencia y conservan su propio código.

3. **Niveles arancelarios:** deriva los niveles de dos y cuatro dígitos, y anula los códigos con letras o de longitud insuficiente, de modo que el potencial de esos grupos se conserve solo en el nivel al que sí corresponde.

4. **Unidades:** el portal reporta en dólares y la cadena trabaja en miles.

5. **Agregación:** suma el potencial por exportador, importador y nivel arancelario.

In [5]:
df_potencial['market_name'] = df_potencial['market_name'].replace(alias_mercados)
df_potencial = df_potencial.merge(df_iso, how='left', on='market_name')
df_potencial['ISO Importador'] = df_potencial['ISO Importador'].fillna(
    df_potencial['market_name'].map(iso_adicionales))
df_potencial['ISO Exportador'] = 'COL'

# el codigo del ITC esta en la nomenclatura HS 2017; se traduce a HS 2012.
# los grupos agregados, que terminan en 'XX' o vienen a cuatro digitos, no tienen
# correspondencia y conservan su propio codigo, igual que en la version anterior
df_potencial = df_potencial.merge(
    df_hs2012_2017, how='left', left_on='hs6_code', right_on='HS6_17'
).drop(columns=['HS6_17'])
df_potencial['HS6'] = df_potencial['HS6'].fillna(df_potencial['hs6_code'])

df_potencial['HS4'] = df_potencial['HS6'].str[:4]
df_potencial['HS2'] = df_potencial['HS6'].str[:2]

# los codigos con letras y los que no llegan a seis digitos no identifican un producto:
# se anulan a ese nivel y el potencial se conserva en el nivel superior
df_potencial['HS4'] = np.where(df_potencial['HS4'].str.contains('[A-Za-z]', na=True), np.nan, df_potencial['HS4'])
df_potencial['HS6'] = np.where(df_potencial['HS6'].str.contains('[A-Za-z]', na=True), np.nan, df_potencial['HS6'])
df_potencial.loc[df_potencial['HS6'].str.len() < 6, 'HS6'] = np.nan

# el portal reporta en dolares; la cadena trabaja en miles
df_potencial['Potencial Exportador'] = df_potencial['export_potential'] / 1_000

df_potencial = (
    df_potencial[['ISO Exportador', 'ISO Importador', 'HS2', 'HS4', 'HS6', 'Potencial Exportador']]
    .dropna(subset=['ISO Importador'])
    .groupby(['ISO Exportador', 'ISO Importador', 'HS2', 'HS4', 'HS6'], dropna=False)['Potencial Exportador']
    .sum().reset_index()
)

temp.mkdir(parents=True, exist_ok=True)
df_potencial.to_parquet(temp / 'potencial_exportador.parquet', index=False)

print(f"{len(df_potencial):,} filas, {df_potencial['ISO Importador'].nunique()} importadores, "
      f"{df_potencial['HS6'].nunique():,} productos HS6")

176,557 filas, 188 importadores, 1,268 productos HS6
